# Avance 4 · Arquitectura en la nube con AWS · FleetLogix
**Proyecto Integrador M2 · Agustina López Torres**

> 🚚 **Torre de control en vivo:** https://claude.ai/artifact/XR7Xm2THf1NywneS81YZYZ

FleetLogix ya tiene su base PostgreSQL con 505 mil registros (Avance 1), consultas optimizadas (Avance 2) y un Data Warehouse en Snowflake (Avance 3). Ahora necesita **recibir y procesar en tiempo real** los datos que mandan los conductores desde sus celulares.

En este avance se **diseña** esa solución en AWS. Como pide la consigna, no se monta la arquitectura real: se explica cómo funciona cada servicio y se prueban las reglas de negocio de las funciones Lambda con Python.

| Parte | Qué resuelve | Servicios |
|---|---|---|
| A | Recibir y guardar los datos | API Gateway, S3, Lambda |
| B | Procesar cada dato | 3 funciones Lambda, SNS, EventBridge |
| C | Bases de datos en la nube | RDS PostgreSQL, DynamoDB, backups |
| Extra | Publicar la API para la app | API Gateway |

## Diagrama de la arquitectura

![Diagrama de arquitectura AWS](aws_architecture_diagram.png)

**Cómo viajan los datos:**
1. La app del conductor manda un mensaje (en formato JSON) por internet al **API Gateway**.
2. API Gateway ejecuta la **función Lambda** que corresponde según la ruta.
3. La Lambda guarda el mensaje en **S3** (histórico), actualiza el estado actual en **DynamoDB** y, si hay un desvío, manda una alerta por **SNS**.
4. Abajo: la base PostgreSQL local se migra a **RDS**. Todas las noches **EventBridge** ejecuta el ETL del Avance 3, que lleva los datos a **Snowflake**.

---
## Parte A · Recibir y guardar datos

### API Gateway: la puerta de entrada
Es la única dirección web que conoce la app. Recibe los pedidos, controla que sean de la app (con una clave llamada API Key), limita cuántos pedidos por segundo acepta y le pasa cada pedido a la Lambda que corresponde.

| Método | Ruta | Cuándo la usa la app | Lambda |
|---|---|---|---|
| POST | `/deliveries/complete` | el conductor cierra una entrega | `verify_delivery` |
| GET | `/trips/eta` | se consulta a qué hora llega el camión | `calculate_eta` |
| POST | `/telemetry/location` | la app manda la posición GPS cada 5 minutos | `route_deviation_alert` |

(POST = la app **manda** datos. GET = la app **pide** un dato.)

### S3: el histórico organizado por fecha
S3 es el almacenamiento de archivos de AWS. Cada mensaje que llega se guarda como un archivo JSON, en carpetas por año, mes y día:

```
s3://fleetlogix-datalake/
├── raw/deliveries/year=2026/month=11/day=16/delivery_2.json
└── raw/telemetry/year=2026/month=11/day=16/trip_1_143500.json
```

Así, si se quiere analizar un día puntual, se lee solo esa carpeta. Configuración del bucket: acceso público bloqueado, cifrado activado, y los archivos de más de 90 días pasan a **Glacier**, una clase de almacenamiento más barata.

### Lambda: pequeños programas que procesan cada dato
Una Lambda es una función de Python que AWS ejecuta **solo cuando llega un dato**. No hay una computadora prendida todo el día, y se paga solo por el tiempo que corre. Si llegan 200 camiones a la vez, AWS ejecuta 200 copias al mismo tiempo.

Las 3 funciones están en la carpeta `lambda_functions/`, con un comentario en cada línea. Todas hacen lo mismo:
1. Leen el JSON que mandó la app.
2. Aplican una regla de negocio.
3. Guardan el resultado en S3 y/o DynamoDB usando **boto3** (la librería de Python para AWS).
4. Le responden a la app.

---
## Parte B · Procesar información

Abajo se prueba la **regla de negocio** de cada Lambda. Son las mismas funciones que están en `lambda_functions/`, sin las líneas de boto3 (que solo funcionan dentro de AWS).

### Lambda 1 · ¿Se completó la entrega?
**Regla:** una entrega está completada si el estado es `delivered`, tiene firma del receptor y tiene fecha de entrega. Los estados son los mismos de la tabla `deliveries`: `delivered`, `failed` y `pending`.

In [1]:
def entrega_completada(entrega):
    estado_ok = entrega.get("delivery_status") == "delivered"   # el estado tiene que ser 'delivered'
    firma = entrega.get("recipient_signature") or ""            # firma del que recibió
    firma_ok = firma.strip() not in ("", "N/A")                 # tiene que haber una firma real
    fecha_ok = bool(entrega.get("delivered_datetime"))          # tiene que tener fecha de entrega
    return estado_ok and firma_ok and fecha_ok                  # las tres cosas a la vez

# Pruebas con entregas de deliveries.csv
pruebas = [
    {"delivery_id": 2, "delivery_status": "delivered", "delivered_datetime": "2025-01-12 00:44:03", "recipient_signature": "Saturnina Vergara"},
    {"delivery_id": 1, "delivery_status": "failed",    "delivered_datetime": "2025-01-12 08:20:47", "recipient_signature": "N/A"},
    {"delivery_id": 9, "delivery_status": "delivered", "delivered_datetime": "2025-01-12 10:00:00", "recipient_signature": ""},
]
for e in pruebas:
    print(f"Entrega {e['delivery_id']}: estado={e['delivery_status']:9} -> completada = {entrega_completada(e)}")

Entrega 2: estado=delivered -> completada = True
Entrega 1: estado=failed    -> completada = False
Entrega 9: estado=delivered -> completada = False


La entrega 9 dice `delivered` pero no tiene firma, así que **no** se da por completada.

### Lambda 2 · Tiempo estimado de llegada (ETA)
**Cálculo:**
1. Distancia en línea recta desde la posición del camión hasta la ciudad de destino (fórmula de Haversine, que mide distancias sobre la Tierra).
2. Se multiplica por 1,25 porque por ruta el camino es más largo que la línea recta.
3. Se divide por **76 km/h**, la velocidad promedio **real** medida en los 100.000 viajes de `trips.csv` (ver la sección del diferencial).

In [2]:
import math

CIUDADES = {   # (latitud, longitud) de las 5 ciudades de la tabla routes
    "Buenos Aires": (-34.6037, -58.3816),
    "Córdoba": (-31.4201, -64.1888),
    "Rosario": (-32.9442, -60.6505),
    "Mendoza": (-32.8895, -68.8458),
    "San Miguel de Tucumán": (-26.8083, -65.2176),
}

def distancia_km(lat1, lon1, lat2, lon2):
    radio = 6371                                                # radio de la Tierra en km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return 2 * radio * math.asin(math.sqrt(a))

def calcular_eta(lat, lon, destino):
    lat_d, lon_d = CIUDADES[destino]                            # coordenadas del destino
    km = distancia_km(lat, lon, lat_d, lon_d) * 1.25            # km que faltan por ruta
    horas = km / 75                                             # tiempo = distancia / velocidad
    return km, horas

# Prueba: camión de Buenos Aires a Córdoba que está pasando por Rosario
km, horas = calcular_eta(-32.9442, -60.6505, "Córdoba")
print(f"Faltan {km:.0f} km, unas {horas:.1f} horas")

Faltan 467 km, unas 6.2 horas


### Lambda 3 · Alerta si el camión se desvía de la ruta
**Cálculo:** la ruta se toma como la línea recta entre la ciudad de origen y la de destino. Se mide la distancia más corta entre el camión y esa línea. Si es **mayor a 50 km**, se manda una alerta por SNS (el servicio de notificaciones de AWS, que la envía por email al centro de control).

¿Por qué 50 km y no menos? Porque la ruta real no es recta: por ejemplo, Rosario está a unos 39 km de la recta Buenos Aires → Córdoba y es un paso normal.

In [3]:
def a_km(lat, lon):
    # pasa latitud y longitud a kilómetros (aproximación válida dentro de Argentina)
    return lon * 111.32 * math.cos(math.radians(-32)), lat * 110.57

def distancia_a_ruta_km(lat, lon, origen, destino):
    px, py = a_km(lat, lon)                                     # el camión
    ax, ay = a_km(*CIUDADES[origen])                            # ciudad de origen
    bx, by = a_km(*CIUDADES[destino])                           # ciudad de destino
    dx, dy = bx - ax, by - ay                                   # dirección de la ruta
    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy) # punto de la ruta más cercano
    t = max(0, min(1, t))                                       # que quede entre origen y destino
    return math.hypot(px - (ax + t * dx), py - (ay + t * dy))   # distancia camión -> ruta

for lugar, lat, lon in [("Marcos Juárez (ruta 9)", -32.70, -62.10),
                        ("Rosario", -32.9442, -60.6505),
                        ("Santa Rosa, La Pampa", -36.62, -64.29)]:
    d = distancia_a_ruta_km(lat, lon, "Buenos Aires", "Córdoba")
    print(f"{lugar:24} -> {d:6.1f} km de la ruta -> {'ALERTA' if d > 50 else 'ok'}")

Marcos Juárez (ruta 9)   ->   12.5 km de la ruta -> ok
Rosario                  ->   38.7 km de la ruta -> ok
Santa Rosa, La Pampa     ->  489.0 km de la ruta -> ALERTA


### Procesamiento automático
Nadie ejecuta nada a mano. Cada servicio dispara al siguiente:

| Cuando pasa esto... | ...se ejecuta solo |
|---|---|
| La app manda un dato a una ruta del API | API Gateway ejecuta la Lambda de esa ruta |
| La Lambda 3 detecta un desvío de más de 50 km | SNS manda el email al centro de control |
| Son las 03:00 (hora Argentina) | EventBridge ejecuta el ETL del Avance 3 (RDS → Snowflake) |
| Una Lambda falla | CloudWatch guarda el error y avisa |
| Un archivo en S3 cumple 90 días | Pasa solo a Glacier |

---
## Parte C · Bases de datos en la nube

### Migrar PostgreSQL local a Amazon RDS
RDS es PostgreSQL, pero administrado por AWS: AWS se encarga del servidor, las actualizaciones y los backups. Al ser el mismo motor, las queries y los índices del Avance 2 funcionan igual.

| Configuración | Valor | Por qué |
|---|---|---|
| Motor | PostgreSQL 16 | El mismo que la base local |
| Tamaño | `db.t4g.micro`, 20 GB | Alcanza para 505 mil registros y entra en la capa gratuita |
| Acceso público | No | Solo acceden las Lambdas y el ETL, desde dentro de AWS |
| Backups automáticos | 7 días | Se puede volver a cualquier momento de la última semana |

**Pasos de la migración:**
1. Crear la base en la consola de AWS: **RDS → Create database → PostgreSQL**, con la configuración de la tabla.
2. Sacar una copia de la base local desde la terminal:
   `pg_dump -h localhost -U postgres -d fleet_logix_bd -F c -f fleet_logix_bd.dump`
3. Cargar esa copia en RDS (el host es la dirección que da AWS al crear la base):
   `pg_restore -h <direccion-de-rds> -U fleet_admin -d fleet_logix_bd fleet_logix_bd.dump`
4. Comprobar que cada tabla tenga la misma cantidad de filas que en la base local:
```sql
SELECT 'vehicles', COUNT(*) FROM vehicles      -- 200
UNION ALL SELECT 'drivers', COUNT(*) FROM drivers       -- 400
UNION ALL SELECT 'routes', COUNT(*) FROM routes         -- 50
UNION ALL SELECT 'trips', COUNT(*) FROM trips           -- 100.000
UNION ALL SELECT 'deliveries', COUNT(*) FROM deliveries -- 399.905
UNION ALL SELECT 'maintenance', COUNT(*) FROM maintenance; -- 5.000
```
5. En el ETL del Avance 3, cambiar `localhost` por la dirección de RDS.

### DynamoDB: el estado actual de las entregas
DynamoDB es una base NoSQL muy rápida. Guarda **una fila por entrega** y la reemplaza con cada dato nuevo, así que siempre tiene solo el estado **actual** (el historial completo está en S3 y RDS). La usamos porque los datos GPS llegan todo el tiempo y la app necesita leer "¿dónde está el camión ahora?" en milisegundos, sin cargar a RDS.

| Tabla | Clave | Qué guarda |
|---|---|---|
| `fleetlogix-delivery-status` | `delivery_id` | estado, si se completó, última actualización |
| `fleetlogix-trip-status` | `trip_id` | última posición, desvío, ETA |

### Backups automáticos

| Dónde | Cómo | Cuánto tiempo |
|---|---|---|
| RDS | Copia automática diaria (03:00) + recuperación a cualquier momento | 7 días |
| DynamoDB | Point-in-Time Recovery (recuperar la tabla a cualquier segundo) | 35 días |
| S3 | Versionado: si un archivo se pisa, queda la versión anterior | Siempre |

Se activan en la consola: RDS → *Backup retention period* = 7 días. DynamoDB → la tabla → *Backups* → *Point-in-time recovery* → activar.

---
## Extra credits · Crear y publicar el API Gateway

Pasos en la consola de AWS:
1. **Crear el API:** API Gateway → *Create API* → *HTTP API* → *Build*. Nombre: `fleetlogix-mobile-api`.
2. **Crear las 3 rutas:** en *Routes*, agregar `POST /deliveries/complete`, `GET /trips/eta` y `POST /telemetry/location`.
3. **Conectar cada ruta a su Lambda:** en cada ruta, *Attach integration* → *Lambda function* → elegir la función correspondiente.
4. **Deploy:** en *Stages*, crear el stage `prod` con *Auto-deploy* activado.
5. **Obtener la URL:** en *Stages → prod* aparece la *Invoke URL*, con esta forma:
   `https://abc123.execute-api.us-east-1.amazonaws.com/prod`

La app la usaría así, por ejemplo para mandar la posición GPS:
```
POST https://abc123.execute-api.us-east-1.amazonaws.com/prod/telemetry/location
{"trip_id": 1, "vehicle_id": 127, "origin_city": "Buenos Aires", "destination_city": "Córdoba", "lat": -32.70, "lon": -62.10}
```

---
## Diferencial · ETA calibrado con los datos y torre de control

### ¿Cuánto tardan de verdad los viajes?
La tabla `routes` tiene una duración estimada para cada ruta. Comparo esa estimación con lo que tardaron de verdad los 100.000 viajes de `trips.csv`.

In [4]:
import pandas as pd

trips = pd.read_csv("../trips.csv", parse_dates=["departure_datetime", "arrival_datetime"])
routes = pd.read_csv("../routes.csv")
viajes = trips.merge(routes, on="route_id")                     # cada viaje con los datos de su ruta

# Duración real de cada viaje, en horas
viajes["horas_reales"] = (viajes["arrival_datetime"] - viajes["departure_datetime"]).dt.total_seconds() / 3600

# ¿Cuántas veces más tarda que lo estimado?
viajes["real_vs_estimado"] = viajes["horas_reales"] / viajes["estimated_duration_hours"]
print("En promedio, un viaje tarda", round(viajes["real_vs_estimado"].mean(), 2), "veces lo estimado")
print("Velocidad promedio real:", round(viajes["distance_km"].sum() / viajes["horas_reales"].sum()), "km/h")

En promedio, un viaje tarda 1.1 veces lo estimado
Velocidad promedio real: 76 km/h


Los viajes tardan en promedio un **10% más** de lo que dice la tabla `routes`. Entonces, ¿mejora el ETA si multiplicamos la estimación por 1,1?

In [5]:
# Error = diferencia entre lo que tardó y lo que se estimó, en minutos
error_tabla      = (viajes["horas_reales"] - viajes["estimated_duration_hours"]).abs().mean() * 60
error_calibrado  = (viajes["horas_reales"] - viajes["estimated_duration_hours"] * 1.1).abs().mean() * 60

print(f"Error usando la duración de la tabla routes: {error_tabla:.0f} minutos")
print(f"Error usando la duración x 1,1:              {error_calibrado:.0f} minutos")

Error usando la duración de la tabla routes: 66 minutos
Error usando la duración x 1,1:              53 minutos


In [6]:
# ¿La hora de salida o el tipo de vehículo cambian algo?
vehicles = pd.read_csv("../vehicles.csv")
viajes = viajes.merge(vehicles[["vehicle_id", "vehicle_type"]], on="vehicle_id")
viajes["hora_salida"] = viajes["departure_datetime"].dt.hour

print("Por tipo de vehículo:")
print(viajes.groupby("vehicle_type")["real_vs_estimado"].mean().round(2))
print("\nPor hora de salida (primeras 6 horas):")
print(viajes.groupby("hora_salida")["real_vs_estimado"].mean().round(2).head(6))

Por tipo de vehículo:
vehicle_type
Camión Grande     1.1
Camión Mediano    1.1
Motocicleta       1.1
Van               1.1
Name: real_vs_estimado, dtype: float64

Por hora de salida (primeras 6 horas):
hora_salida
0    1.1
1    1.1
2    1.1
3    1.1
4    1.1
5    1.1
Name: real_vs_estimado, dtype: float64


**Conclusión:** calibrar el ETA con los datos reales **baja el error de 66 a 53 minutos**. La hora y el tipo de vehículo no cambian la duración (siempre da 1,10), así que un modelo más complejo no mejoraría el resultado con estos datos. Por eso la Lambda `calculate_eta` usa la velocidad real medida (76 km/h). Si en el futuro hubiera datos reales con patrones de tráfico, ahí sí convendría entrenar un modelo de machine learning.

### Torre de control en vivo
Para mostrar cómo funcionaría todo junto, armé una **torre de control** (`torre_de_control.html`, se abre con doble clic en el navegador):
- Un mapa con las 5 ciudades y camiones recorriendo viajes reales de `trips.csv`, con su patente, tipo de vehículo y conductor.
- Al tocar un camión se ve su ETA y el mensaje que mandaría la app a `POST /telemetry/location`.
- Si un camión se aleja más de 50 km de su ruta, se pone en rojo y aparece la alerta. Hay un botón para simular un desvío en vivo.
- Al llegar cada camión se compara el error del ETA con y sin calibrar.

**Verla en vivo:** https://claude.ai/artifact/XR7Xm2THf1NywneS81YZYZ

---
## Conclusiones
- **Sin servidores prendidos:** API Gateway y Lambda se pagan solo cuando se usan y crecen solos si la flota crece.
- **Cada dato en su lugar:** S3 guarda el histórico (barato), DynamoDB el estado actual (rápido), RDS las 6 tablas relacionales y Snowflake el análisis (Avance 3).
- **Automático:** la app dispara las Lambdas, las Lambdas disparan las alertas y EventBridge dispara el ETL nocturno.
- **Seguro y con respaldo:** solo la API es pública, todo va cifrado y hay backups automáticos en RDS, DynamoDB y S3.
- **Con datos:** el ETA se calibró con los viajes reales y el error bajó de 66 a 53 minutos.
- **Costo estimado:** unos USD 55 por mes para 200 vehículos (detalle en `Arquitectura_Tecnica.pdf`).